# STAT163 · Practice 3 — from an event log to a description of visitors

One notebook across your two practice sessions. Submit the changes in it.

Each part has one or two **Task** cells with a self-check under them. They are the
required minimum, and the text before each one tells you what to build. Each part ends
with an **Explore and discuss** block: optional calculations that go further, and the
questions to argue about in the room. The note in Part 6 is where you comment on what you
found, and it is graded.

Graded: the Task cells, the note, and the disclosure cell at the end. A Task cell earns
its marks by holding an answer that runs. The note earns its marks by commentary that the
numbers in your notebook support. The note draws on the Explore blocks, so do at least
the first prompt of each: without them you have little to comment on.

Some Task cells hold a line that starts with `#`. Remove the `#` and complete the line;
it is commented out so that the notebook runs before you fill it in.

**AI in this practice: recommended mode.**

Try each task yourself first.

If you are stuck in the classroom, ask the teacher or TA. Working remotely: ask your LLM to
explain the concept rather than to solve. Two boundaries are firm:

- say what you used, in the disclosure cell at the end
- do not submit a notebook authored end to end by AI. It is not your work and it is not
  accepted.

## Before you start

Replace the placeholder with your real name and run the cell.

In [ ]:
from selfcheck import check, check_col, check_identity

student_name = "Name Surname"   # ← replace with your name

check_identity(student_name)

## Part 1 — Profiling

A web shop recorded what its visitors did for four and a half months of 2015: one row each
time a visitor did one thing to one item. Published on Kaggle as the
**[Retailrocket recommender system dataset](https://www.kaggle.com/datasets/retailrocket/ecommerce-dataset)**.

| Column | What it holds |
|---|---|
| `timestamp` | when, as a whole number: milliseconds since 1 January 1970, the way many systems store time |
| `visitorid` | who, a number the shop gave the visitor's browser |
| `event` | what: `view`, `addtocart` or `transaction` |
| `itemid` | which item |
| `transactionid` | the purchase the row belongs to, on `transaction` rows only |

`pd.to_datetime` reads the whole number when told the unit. The load cell adds a `time`
column that way and keeps `timestamp` as it came.

In [ ]:
import pandas as pd

events_raw = pd.read_csv("data/events.csv.gz")
events_raw["time"] = pd.to_datetime(events_raw["timestamp"], unit="ms")

events_raw.head()

Before doing any analysis, we need to profile a new, unknown dataset. Until we understand
its shape and its granularity, whether we need to change data types, what to do with
missing values, and whether there are duplicates to remove, we cannot be sure about the
quality of anything we compute on it.

### describe()

Use `describe` on every column of the dataframe. Notice what it gives you for the numeric
columns and for the text columns, and notice how the date column behaves.

Not every value in the `describe()` output makes sense, but some of them are diagnostic.
Take a look at the values and see what you can already guess about the dataframe.

- Compare the number of rows with the `count` value of each column. How would you
  interpret the difference?
- Take a look at the `top` and `freq` values for the `event` column. What do they tell you?
- `time` column: look at the `min` and `max` values.
- `visitorid`, `transactionid`, `itemid`: does it make sense to interpret them as numbers
  here? Which additional checks would you want to do to describe these columns?

In [ ]:
print(events_raw.shape)
events_raw.describe(include="all")

Calculate the descriptive values for `visitorid`, `itemid` and `transactionid` that
`describe()` did not give you: the number of unique values, the top value and the
frequency of the top value.

There are at least two ways to do this: convert the columns to text and run `describe`
again, or apply the methods you already know to each column to get the analogues of
`unique`, `top` and `freq`.

In [ ]:
# Sandbox

### Types and missing values

In [ ]:
events_raw.dtypes

Take a look at the data types of `transactionid` and of the other "numeric" id columns.
Why is it `float64` while the rest are `int64`?

Try to convert it to integer with `astype`. Remove the `#` in the cell below and run it:
it stops with an error. Read the error and figure out what causes it, then put the `#`
back, so that the whole notebook can run from top to bottom.

In [ ]:
# events_raw["transactionid"].astype("int64")

Now calculate the number of missing values in each column and confirm what you guessed
from `describe()`.

*You may need:* `isna`, `sum`.

In [ ]:
# Sandbox

### Duplicates

Rows that repeat another row in every column:

In [ ]:
events_raw.duplicated().sum()

You get 152 rows: the same visitor, the same action, the same item, the same millisecond.
No human can produce two events in the same millisecond, so these are almost certainly
duplicates and it is safe to remove them.

For some calculations this small number would not matter. But to be sure you do not carry
duplicate rows into everything you build from this table, it is always better to drop
them now. From here on we work with `events`.

In [ ]:
events = events_raw.drop_duplicates()

print(len(events_raw), len(events))

### Consistency

When the technical checks are done and you have a general understanding of the data, the
next step of profiling is to look for internal consistency: does the table keep the rules
it seems to promise?

In this dataset the transactions are a small but important subset, and we want to be sure
we see no problems there. We have an event called `transaction` and a column
`transactionid`, and the table in the introduction says one goes with the other. Two
hypotheses to test:

- **Hypothesis 1:** every `transaction` event has a `transactionid`, and no other event
  has one.
- **Hypothesis 2:** each transaction belongs to one visitor and happens at one moment.

Test the first one by counting the rows that would break it, in both directions. If the
hypothesis holds, both counts are 0.

*You may need:* `.loc` with a condition and a column name, `isna`, `notna`, `sum`.

In [ ]:
# Sandbox

In [ ]:
# Task 1.1 — test Hypothesis 1. Count the transaction rows with an empty transactionid,
# and the rows of the other two kinds with a filled transactionid.
n_transaction_rows_without_id = ...
n_other_rows_with_id = ...

check("n_transaction_rows_without_id", n_transaction_rows_without_id, "integer")
check("n_other_rows_with_id", n_other_rows_with_id, "integer")

So the id and the event go together. Now count how many transactions there are. Two
methods on the same column give two different numbers: `count` gives the rows that carry
a value, `nunique` the distinct values.

In [ ]:
# Sandbox

In [ ]:
# Task 1.2 — the rows with event == "transaction", and the distinct transaction ids
n_transaction_rows = ...
n_transactions = ...

check("n_transaction_rows", n_transaction_rows, "integer")
check("n_transactions", n_transactions, "integer")

The two numbers differ, so a transaction can take several rows. The second hypothesis is
about those rows: do they belong to one visitor, and do they share one moment? A groupby
on `transactionid` drops the rows with no id, so it works on the transaction rows alone.

In [ ]:
per_transaction = events.groupby("transactionid").agg(
    rows=("visitorid", "size"),
    visitors=("visitorid", "nunique"),
    moments=("timestamp", "nunique"),
)

print(per_transaction["visitors"].max(), per_transaction["moments"].max())
per_transaction.sort_values("moments", ascending=False).head()

One half of the hypothesis holds and the other does not. The largest number of visitors
in a transaction is 1, so every transaction has one visitor. The largest number of moments
is far above 1, and the table shows the transactions with the most: a transaction is not
one instant in this log.

This is the point where profiling starts a second round. Once we look at the transactions
as a table of their own, we have new assumptions and new questions about it: what is one
row of that table, and does it hold the same kind of duplicates the whole log did? We do
not compute anything on the transactions yet, but the first block below takes that up.

### Explore and discuss

**Grain.** We established that one row of `events` is one event. A subset of the table
can be described more precisely than the whole. Build the `transactions` table, the rows
with `event == "transaction"`, and profile it on its own: what does one row represent
there? Check whether a pair of `transactionid` and `itemid` repeats: `duplicated` takes
a list of columns, and `keep=False` marks every copy, not only the later ones. If pairs
repeat, look at the timestamps of the repeats and decide what they are: a second unit of
the item, or the same purchase logged twice. Are there opportunities to clean this table
that the whole log did not show?

**Active hours, if time allows.** The log holds whole days, in UTC. Group the events by
the hour of `time`, with `events["time"].dt.hour` as the key of the groupby, and look at
the counts. At what time of day do most visitors come to the store? The shop's own clock
is not UTC: read the hour of the first and the last row of the log. If the log holds whole
days on the shop's clock, at which UTC hour does the shop's day start?

**Discuss:** say in one sentence what one row of `events` is, and in one more what one row
of `transactions` is. That first sentence opens the note in Part 6.

In [ ]:
# Explore

## Part 2 — From events to visitors

Every question about a visitor is a question about their rows: how many they have, and how
many of each kind. The table we need should have one row per visitor, and we get there by grouping
the events on `visitorid`.

To count each kind of row inside a group, first you need to store each kind as a True/False column:
`is_view` is True on the view rows, and so on. A groupby can then sum such a column, and
the sum is the number of rows of that kind.

`agg` takes a name for each result column, then the column to aggregate and the
aggregation. Leave `visitorid` as the index of the result: the blocks that follow select
visitors by it.

*You may need:* a comparison per condition; `groupby`, `agg` with `"size"`, `"sum"` and
`"nunique"`.

In [ ]:
# Sandbox

In [ ]:
# Task 2.1 — three True/False columns on events: is_view (view rows), is_cart (addtocart
# rows), is_purchase (transaction rows).
events["is_view"] = ...
events["is_cart"] = ...
events["is_purchase"] = ...

# Then `visitors`, one row per visitorid:
#   events     rows
#   views      view rows
#   carts      addtocart rows
#   purchases  transaction rows
#   items      distinct items, across all kinds of row
visitors = ...

check_col(events, "is_view")
check_col(events, "is_cart")
check_col(events, "is_purchase")
check("visitors", visitors, "DataFrame")

Before you go on, compare the length of `visitors` with the number of distinct visitors
from Part 1. They must be equal: a table with the wrong number of rows gives wrong answers
to everything below, and no error.

### Explore and discuss

**The busiest visitor.** Sort `visitors` by `events` and read the top rows: events,
purchases, distinct items. Is the top row a person?

**Distributions.** Values in the columns of `visitors` table have
distributions, and they describe the visitors better than any single number. This may
sound like an aggregation of an aggregation, but it is not: a share of visitors with some
property is a description of the visitors, not an average of averages. Calculate:

- the share of visitors with exactly one event;
- the share of visitors with at least one purchase;
- the share of visitors with more than 10 views;
- the share of visitors who used the cart, and among them the share who bought;
- the mean and the median number of events per visitor.

**Discuss:** the mean and the median are two answers to "a typical visitor". Which one
goes in the note, and what stands next to it so that the reader knows what it hides?

In [ ]:
# Explore

## Part 3 — Monthly Totals


Make a table of monthly statistics: total views, total add-to-carts, total purchases.
Then calculate the ratios between them. In marketing these are called **conversions**:
view-to-cart, cart-to-purchase and view-to-purchase. Each one is two columns of the same
table divided row by row.

Note: It is convenient to create a new column when you are going to group on it often, but it is
not required: the key of a groupby can be a Series made on the spot. Every row is in
2015, so the month number is enough, and `events["time"].dt.month` gives it.

*You may need:* `groupby` with a Series as the key, `agg`.

In [ ]:
# Sandbox

In [ ]:
# Task 3.1 — `monthly`, one row per month with the month number as the index:
#   views      view rows
#   carts      addtocart rows
#   purchases  transaction rows
# Then three columns: view_to_cart (carts divided by views), cart_to_purchase (purchases
# divided by carts), view_to_purchase (purchases divided by views).
monthly = ...
# monthly["view_to_cart"] = ...
# monthly["cart_to_purchase"] = ...
# monthly["view_to_purchase"] = ...

check("monthly", monthly, "DataFrame")
check_col(monthly, "view_to_cart")
check_col(monthly, "cart_to_purchase")
check_col(monthly, "view_to_purchase")

Read the table with the counts beside the ratios. One month looks different from the
others; before you call that a finding, look at the first and the last `time` in the log.

### Explore and discuss

**Change the grain of the calculation.** The totals and the ratios above count events.
A visitor who viewed one item thirty times adds thirty to the views, although they are
one person who looked and did not buy. It is often more useful to compute these ratios
per visitor: each visitor counts once per month for viewing, once for putting something
in the cart, once for buying. Compare the two tables when you have them: do the ratios
move much on this log, and does that change which one you would report?

One way to get there: group by the month and `visitorid` together, and take the `max` of
each condition column, which is True when the visitor did that at least once in the
month. That table has one row per visitor and month, and its index has two levels. A
groupby key can be the name of an index level as well as a column, so a second groupby on
the month level sums the table up to one row per month.

Add `visitors`, the distinct visitors per month, to the same table. Then add the monthly
visitor counts up and compare the sum with the distinct visitors from Part 1. The sum is
larger, because a visitor who came in several months is counted once in each of them: a
distinct count for the whole period is computed on the whole period, and the monthly
counts do not add into it.

**Discuss:** what are the advantages and the disadvantages of each way of counting, per
event and per visitor? Which one would you put in the note, and what stands next to it?

In [ ]:
# Explore

## Part 4 — New and returning visits

Some visitors come to the store for the first time, some are returning after a while. We
can tell the two apart from the data we have, by the time of each event they had.

One way to do it is to calculate the first visit time of each visitor and compare every
event to it. The rule for a returning visit could be: it happened a full day or more
after the first visit.

To get there, first calculate a visitor-level `first_seen` column on every row of that
visitor, with `transform`. Then subtract it from each event's `time`. The result is a
column of a special data type, `timedelta`, a duration: explore what it can do and find
the way to present it as a number of whole days. When the number of days is 1 or more,
the event is a returning visit.

*You may need:* `groupby`, `transform`; a duration column has `.dt.days`.

In [ ]:
# Sandbox

In [ ]:
# Task 4.1 — first_seen: the earliest time of the row's visitor, on every row of that
# visitor. Then days_since_first: whole days from first_seen to time. Then is_returning:
# True when days_since_first is 1 or more.
events["first_seen"] = ...
# events["days_since_first"] = ...
# events["is_returning"] = ...

check_col(events, "first_seen")
check_col(events, "days_since_first")
check_col(events, "is_returning")

Now every event is either a first-day visit or a returning one, and `is_returning` is a
condition like the ones in Part 2: its mean is a share, its sum is a count, and it can be
summed per group.

### Explore and discuss

**New and returning visits.** What share of all events are returning visits? How many
visitors have at least one returning visit, and what share of all visitors is that?

**New and returning purchases.** Now the same for the transaction rows: what share of
purchases happen on a returning visit? Then compare the buyers: among the visitors who
ever returned, what share bought at least once, against the same share among the
visitors who never returned? Put the size of each group beside its share.

**Discuss:** the log starts on 3 May. A visitor whose first row is on 4 May may have been
coming to this shop for a year, and we call their visit new. The split you computed is
limited by the period we have: the correct one needs the full history since the start of
sales. Say what that does to each number above, and in which direction. Then, if
returning visitors buy more often, give two different explanations that would both
produce that number, and say whether the log can tell them apart.

In [ ]:
# Explore

## Part 5 — From a view to a purchase

This part looks at visitors and the items they buy. **The goal** is to calculate, for
each pair of a visitor and an item, the time of the view, the time of the add-to-cart and
the time of the transaction, and then the time it takes between each step.

Take into account that a visitor and an item quite often have several events of the same
kind: people come back to an item and view it again, or add it to the cart twice. Check
whether even a transaction can repeat. To get a meaningful analysis of time we need one
moment per kind of event, and the natural choice is the earliest one: the first view, the
first add-to-cart, the first purchase.

The preparation cell puts the time of each event into a column of its own kind. `where`
keeps the time where the condition is True and puts a missing value elsewhere, so
`view_time` is filled only on the view rows, and so on.

In [ ]:
# preparation: one time column per kind of event
events["view_time"] = events["time"].where(events["event"] == "view")
events["cart_time"] = events["time"].where(events["event"] == "addtocart")
events["purchase_time"] = events["time"].where(events["event"] == "transaction")

events[["visitorid", "itemid", "event", "view_time", "cart_time", "purchase_time"]].head()

Now group by two keys, `visitorid` and `itemid` together. The result has one row per
combination that occurs, with both keys as the index. `min` skips missing values, so a
pair that was never bought gets a missing `first_purchase`.

*You may need:* `groupby` with a list of two keys, `agg` with `"min"`.

In [ ]:
# Sandbox

In [ ]:
# Task 5.1 — `pairs`, one row per visitorid and itemid:
#   first_view      earliest view_time
#   first_cart      earliest cart_time
#   first_purchase  earliest purchase_time
pairs = ...

check("pairs", pairs, "DataFrame")

`pairs["first_purchase"].notna()` marks the bought pairs. Compare their number with the
transaction rows from Part 1. The difference is the rows that the earliest-event rule
folded into one: the same visitor buying the same item again, and the repeated rows you
found in the transactions in Part 1.

The durations between the steps are subtractions of two time columns. Where one of the
two is missing, the result is missing too, so a pair that was viewed and never bought gets
no `view_to_purchase`. In Part 4 we measured time from a visitor's first moment of any
kind; here the clock starts at the first view of this item.

In [ ]:
# Sandbox

In [ ]:
# Task 5.2 — three duration columns on pairs: view_to_cart (first_view to first_cart),
# cart_to_purchase (first_cart to first_purchase), view_to_purchase (first_view to
# first_purchase).
# pairs["view_to_cart"] = ...
# pairs["cart_to_purchase"] = ...
# pairs["view_to_purchase"] = ...

check_col(pairs, "view_to_cart")
check_col(pairs, "cart_to_purchase")
check_col(pairs, "view_to_purchase")

### Explore and discuss

**How long each step takes.** The median of each duration column, in minutes:
`.dt.total_seconds() / 60` reads a duration in minutes, and the median skips the missing
values. Put beside each median the number of pairs it was computed on. Some durations come
out negative: what does a negative `cart_to_purchase` mean, and what do you do with it?

**Purchases that skipped the cart.** Of the bought pairs, what share had the item in the
cart before the purchase? A comparison with a missing value is False, so
`pairs["first_cart"] < pairs["first_purchase"]` is True only where both exist and the
cart came first. What about the pairs that were bought with no view at all: what does
that say about what the log records?

**Carts left behind.** Of the pairs with a `first_cart`, what share was never bought
afterwards? Count the pairs with a `first_purchase` after the cart, and the rest are
the carts left behind.

**The item that converts best, if time allows.** One row per item, built the way
`visitors` was: `views` and `purchases` as sums of the conditions, and purchases divided
by views as the conversion. An item with purchases and no views divides by zero, and
pandas writes `inf` without an error. Choose a minimum number of views before you sort,
name the winner with its views and purchases, and say why that threshold.

**Discuss:** do visitors who use the cart buy more often than those who do not?
`visitors["carts"] > 0` is the cart-user condition. Say what one row of your answer
stands for before you compute it.

In [ ]:
# Explore

## Part 6 — The note

The note is what you would send to someone who asks what this shop's visitors do. It is
not a list of the numbers you computed: it is your commentary on what you found, and the
numbers stand beside the commentary to support it. A count next to every rate, the group
sizes next to every comparison, and the mean or the median named when you use one.

The first line is written as a model. Write the rest in the same way: one or two
sentences per line, a finding first, the numbers that support it after. Double-click the
cell to edit it, then run it with Shift+Enter.

> **What one row is.** One row is one event: a visitor viewing, adding to the cart or
> buying one item at one moment. Most of the log is looking: 915 thousand events from
> 469 thousand visitors, and 97% of the events are views.
>
> **A typical visitor.** *(the mean or the median, and what it hides)*
>
> **Who buys.** *(the share of buyers, and what separates them from the rest)*
>
> **By month.** *(the conversions, per event or per visitor, and the month that differs)*
>
> **New and returning.** *(visits, purchases, and what the period we have does to it)*
>
> **From a view to a purchase.** *(how long each step takes, and the steps the log misses)*
>
> **What the log cannot tell.** *(at least two questions this notebook asked that the log cannot answer)*

## Before you submit

**Say what you used.** One line naming the tool and the step it helped with, or
"No AI used".

> *Example: "Used Claude to see why purchases divided by views showed inf."*

In [ ]:
ai_disclosure = "..."

check("ai_disclosure", ai_disclosure, "text")

**Submit:** restart and re-run the whole notebook (*Kernel → Restart Kernel and Run All
Cells*), check that every cell runs, save, commit and push. Then paste your repository
URL into the Week 3 practice assignment on Moodle.

The deadline is shown on the Moodle assignment.